# 4.3 — Approfondissement : recherche dense et hybride (guide, annexe C.1)

**Question posée.** REC-v2 (BM25 + racinisation) laisse deux questions DEV sans preuve dans le top-5 : Q012
(« nourri par sonde » / « nutrition entérale ») et Q016 (signes d'infection), dont les mots diffèrent de ceux du
passage. Une recherche sémantique retrouve-t-elle ces preuves, sans dégrader les autres, et à quel coût ?

**Statut.** Approfondissement, après le socle : Q0/Q1/Q2 restent définis avec la recherche lexicale (livret § 4).
Les réglages se font sur DEV seulement ; TEST n'est pas interrogé.

**Méthodes (code du TD RAG, `src/qa_recherche.py`)** :
- **dense** : `intfloat/multilingual-e5-small`, préfixes `passage:` / `query:`, vecteurs normalisés, cosinus ;
- **hybride** : fusion des rangs (RRF) des 20 premiers BM25 REC-v2 et des 20 premiers denses, `1 / (60 + rang)` ;
  les scores bruts ne sont pas additionnés.

In [1]:
import sys, json, time
from pathlib import Path

import pandas as pd

RACINE = Path.cwd().parent
sys.path.insert(0, str(RACINE))

from src.qa_recherche import (construire_bm25, scores_bm25, premiers, mesures_question, IndexDense, fusion_rrf,
                              MODELE_DENSE, K1, B)

DATA = RACINE / "data"
RES = RACINE / "resultats" / "qa"
K = 5

passages = [json.loads(l) for l in (DATA / "passages.jsonl").read_text(encoding="utf-8").splitlines()]
dev = [json.loads(l) for l in (DATA / "questions_dev.jsonl").read_text(encoding="utf-8").splitlines()]
ids = [p["passage_id"] for p in passages]
textes = [p["texte"] for p in passages]
sections = [f'{p["localisation"]["section"]} {p["texte"]}' for p in passages]

## Index

BM25 = REC-v2 (texte, racinisation, k1 = 1,5, b = 0,75). Index dense encodé sur le texte seul et sur titre de section
+ texte (le modèle voit le contexte du passage) ; temps d'encodage mesuré sur CPU.

In [2]:
bm25 = construire_bm25(textes, racines=True)

denses, couts = {}, {}
for nom, corpus in {"texte": textes, "section + texte": sections}.items():
    debut = time.perf_counter()
    denses[nom] = IndexDense(corpus)
    couts[nom] = time.perf_counter() - debut
    print(f"dense « {nom} » : {denses[nom].vecteurs.shape}, {couts[nom]:.1f} s (chargement + encodage, CPU), "
          f"passages tronqués : {denses[nom].tronques}")

/home/pico-dev/venvs/nlp/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


dense « texte » : (413, 384), 35.5 s (chargement + encodage, CPU), passages tronqués : 0


dense « section + texte » : (413, 384), 29.8 s (chargement + encodage, CPU), passages tronqués : 0


In [3]:
def classement_bm25(question):
    return [i for i, _ in premiers(scores_bm25(question, bm25, K1, B), len(passages))]

def classement_dense(question, nom):
    return [i for i, _ in premiers(denses[nom].scores(question), len(passages))]

METHODES = {
    "BM25 REC-v2": classement_bm25,
    "dense (texte)": lambda q: classement_dense(q, "texte"),
    "dense (section + texte)": lambda q: classement_dense(q, "section + texte"),
    "hybride RRF (BM25 + dense texte)": lambda q: [i for i, _ in fusion_rrf(
        [classement_bm25(q), classement_dense(q, "texte")], k=len(passages))],
    "hybride RRF (BM25 + dense section + texte)": lambda q: [i for i, _ in fusion_rrf(
        [classement_bm25(q), classement_dense(q, "section + texte")], k=len(passages))],
}

classements, durees = {}, {}
for nom, methode in METHODES.items():
    debut = time.perf_counter()
    classements[nom] = {q["question_id"]: [ids[i] for i in methode(q["question"])] for q in dev}
    durees[nom] = time.perf_counter() - debut

lignes = [{"methode": nom, "question_id": q["question_id"], **mesures_question(q, classements[nom][q["question_id"]], K)}
          for nom in METHODES for q in dev if q["answerable"]]
detail = pd.DataFrame(lignes)
MESURES = [f"hit@{K}", f"rappel@{K}", f"hit@{K}_equiv", f"rappel@{K}_equiv", f"couverture_elements@{K}"]
comparaison = detail.groupby("methode", sort=False)[MESURES].mean().round(3)
comparaison["recherche 20 questions (s)"] = pd.Series(durees).round(2)
comparaison.to_csv(RES / "recherche_dev_hybride.csv")
comparaison

,hit@5,rappel@5,hit@5_equiv,rappel@5_equiv,couverture_elements@5,recherche 20 questions (s)
methode,,,,,,
BM25 REC-v2,0.875,0.781,0.875,0.812,0.823,0.08
dense (texte),0.812,0.719,0.812,0.750,0.760,0.91
dense (section + texte),0.938,0.875,1.000,0.969,0.979,0.74
hybride RRF (BM25 + dense texte),0.812,0.656,0.812,0.688,0.698,0.70
hybride RRF (BM25 + dense section + texte),0.875,0.781,0.875,0.781,0.802,1.01


In [4]:
# Rang de la première preuve par question (> 5 : hors top-5 ; les ex æquo de RRF au-delà des 20 candidats sont sans rang).
rangs = detail.pivot(index="methode", columns="question_id", values="rang_premiere_preuve").loc[list(METHODES)]
rangs

question_id,Q001,Q002,Q003,Q004,Q006,Q007,Q008,Q009,Q010,Q011,Q012,Q013,Q014,Q015,Q016,Q017
methode,,,,,,,,,,,,,,,,
BM25 REC-v2,1,3,1,1,1,1,5,1,2,1,22,1,5,1,140,2
dense (texte),1,2,4,1,2,1,1,85,4,1,2,1,2,1,14,42
dense (section + texte),3,2,2,1,14,1,1,1,3,1,4,1,1,1,4,1
hybride RRF (BM25 + dense texte),1,2,3,1,1,1,1,9,3,1,5,1,2,1,19,7
hybride RRF (BM25 + dense section + texte),2,2,1,1,2,1,1,1,5,1,7,1,1,1,15,1


In [5]:
# Top-5 des deux questions visées et de deux questions bien traitées par BM25, pour vérifier qu'on ne dégrade rien.
textes_par_id = {p["passage_id"]: p for p in passages}
for qid in ["Q012", "Q016", "Q010", "Q001"]:
    q = next(q for q in dev if q["question_id"] == qid)
    print(f"── {qid} — {q['question']}\n   preuves : {q['evidence_ids']}")
    for nom in ["BM25 REC-v2", "hybride RRF (BM25 + dense texte)"]:
        top = classements[nom][qid][:K]
        print(f"   {nom:34s}: " + ", ".join(f"{i}{'*' if i in q['evidence_ids'] else ''}" for i in top))
    print()

── Q012 — Mon conjoint va être nourri par sonde à la maison : comment doit-il être installé, combien de temps dure le passage et qui s'occupe du matériel ?
   preuves : ['DOC008_P027', 'DOC008_P028']
   BM25 REC-v2                       : DOC008_P031, DOC008_P026, DOC001_P110, DOC005_P013, DOC008_P025
   hybride RRF (BM25 + dense texte)  : DOC008_P026, DOC008_P025, DOC008_P031, DOC001_P058, DOC008_P028*

── Q016 — Quand les globules blancs sont bas après une chimiothérapie, quels signes doivent faire penser à une infection et que faut-il faire ?
   preuves : ['DOC001_P118']
   BM25 REC-v2                       : DOC001_P100, DOC001_P139, DOC001_P112, DOC001_P044, DOC001_P102
   hybride RRF (BM25 + dense texte)  : DOC001_P112, DOC001_P139, DOC001_P113, DOC001_P115, DOC001_P102

── Q010 — Que manger et que boire en cas de diarrhée pendant la chimiothérapie, et quand faut-il alerter son médecin ?
   preuves : ['DOC007_P008', 'DOC007_P009']
   BM25 REC-v2                       : DOC001_P08

In [6]:
# Questions sans réponse : le passage classé premier par chaque méthode (ce que recevrait Q0).
pd.DataFrame({nom: {q["question_id"]: f'{classements[nom][q["question_id"]][0]} — '
                                      f'{textes_par_id[classements[nom][q["question_id"]][0]]["localisation"]["section"][-45:]}'
                    for q in dev if not q["answerable"]} for nom in ["BM25 REC-v2", "hybride RRF (BM25 + dense texte)"]})

,BM25 REC-v2,hybride RRF (BM25 + dense texte)
Q005,DOC003_P011 — icaments pour limiter les effets...,DOC003_P011 — icaments pour limiter les effets...
Q018,DOC001_P174 — Fiche 2. Le casque réfrigérant,DOC001_P174 — Fiche 2. Le casque réfrigérant
Q019,DOC001_P117 — nution temporaire de certains gl...,DOC001_P154 — vacances lorsque le traitement ...
Q020,DOC005_P016 — onseils alimentaires pendant la ...,DOC001_P039 — édicaments prescrits pour d’autr...


## Essais de fusion favorisant la recherche dense (2026-10-29)

La fusion à poids égaux ne dépasse pas la recherche dense seule : quand une méthode rate complètement une preuve,
la moyenne des rangs la fait reculer. Deux variantes ont été essayées, toujours avec BM25 REC-v2 et la recherche dense
sur titre de section + texte (meilleure composante dense) :

- **RRF pondérée** : la contribution dense est multipliée par un poids (1 ; 1,5 ; 2 ; 3) ;
- **règle « grand écart »** : si le premier passage d'une méthode n'est pas dans les 20 premiers de l'autre, on suit
  la recherche dense seule ; sinon, on fusionne à poids égaux.

Ces règles ont été imaginées après avoir vu les échecs de DEV : leur résultat est donc optimiste, et elles ajoutent un
paramètre réglé sur 16 questions. Elles ne sont retenues que si elles dépassent nettement la recherche dense seule,
qui n'a aucun paramètre.

In [7]:
def fusion_ponderee(question, poids_dense):
    return [i for i, _ in fusion_rrf([classement_bm25(question), classement_dense(question, "section + texte")],
                                     k=len(passages), poids=[1.0, poids_dense])]

def regle_grand_ecart(question, seuil=20):
    cb, cd = classement_bm25(question), classement_dense(question, "section + texte")
    desaccord = cd[0] not in cb[:seuil] or cb[0] not in cd[:seuil]
    return cd if desaccord else [i for i, _ in fusion_rrf([cb, cd], k=len(passages))]

ESSAIS = {"BM25 REC-v2": classement_bm25,
          "dense (section + texte)": lambda q: classement_dense(q, "section + texte")}
ESSAIS.update({f"RRF, poids dense × {w}": (lambda q, w=w: fusion_ponderee(q, w)) for w in [1, 1.5, 2, 3]})
ESSAIS["règle « grand écart » → dense"] = regle_grand_ecart

lignes = [{"methode": nom, "question_id": q["question_id"], **mesures_question(q, [ids[i] for i in f(q["question"])], K)}
          for nom, f in ESSAIS.items() for q in dev if q["answerable"]]
essais = pd.DataFrame(lignes)
resume_essais = essais.groupby("methode", sort=False)[[f"hit@{K}", f"rappel@{K}", f"couverture_elements@{K}"]].mean().round(3)
resume_essais["paramètres réglés"] = [0, 0, 0, 1, 1, 1, 1]
resume_essais.to_csv(RES / "recherche_dev_fusion_essais.csv")
resume_essais

,hit@5,rappel@5,couverture_elements@5,paramètres réglés
methode,,,,
BM25 REC-v2,0.875,0.781,0.823,0
dense (section + texte),0.938,0.875,0.979,0
"RRF, poids dense × 1",0.875,0.781,0.802,0
"RRF, poids dense × 1.5",0.938,0.812,0.844,1
"RRF, poids dense × 2",0.938,0.812,0.844,1
"RRF, poids dense × 3",0.938,0.812,0.844,1
règle « grand écart » → dense,0.875,0.781,0.885,1


In [8]:
# Rang de la première preuve par question : la fusion pondérée récupère Q006 (aphtes, bien vu par BM25)
# mais fait reculer Q012 et Q016, que seule la recherche dense trouvait.
essais.pivot(index="methode", columns="question_id", values="rang_premiere_preuve").loc[list(ESSAIS)]

question_id,Q001,Q002,Q003,Q004,Q006,Q007,Q008,Q009,Q010,Q011,Q012,Q013,Q014,Q015,Q016,Q017
methode,,,,,,,,,,,,,,,,
BM25 REC-v2,1,3,1,1,1,1,5,1,2,1,22,1,5,1,140,2
dense (section + texte),3,2,2,1,14,1,1,1,3,1,4,1,1,1,4,1
"RRF, poids dense × 1",2,2,1,1,2,1,1,1,5,1,7,1,1,1,15,1
"RRF, poids dense × 1.5",3,2,1,1,3,1,1,1,5,1,5,1,1,1,13,1
"RRF, poids dense × 2",3,2,1,1,3,1,1,1,5,1,5,1,1,1,13,1
"RRF, poids dense × 3",3,2,1,1,3,1,1,1,5,1,5,1,1,1,13,1
règle « grand écart » → dense,2,2,2,1,14,1,1,1,3,1,7,1,1,1,4,1


**Conclusion.** Aucune variante de fusion ne dépasse la recherche dense seule (Rappel@5 0,812 au mieux contre
0,875), et toutes ajoutent un paramètre réglé sur DEV. La règle « grand écart » revient à suivre la recherche dense
quand elle est meilleure, puis perd des rangs quand la fusion reprend la main. Pour l'approfondissement, on retient
donc la **recherche dense sur titre de section + texte, sans fusion** ; BM25 REC-v2 reste le socle.

## Top-5 de la recherche dense pour Q2 (DEV) → `resultats/qa/recherche_dense_top5_dev.jsonl`

Même format que les passages retrouvés de Q0 ; utilisé par `4.2_llm_q1_q2.ipynb` pour mesurer la qualité des réponses
de Q2 avec la recherche dense (approfondissement).

In [9]:
with open(RES / "recherche_dense_top5_dev.jsonl", "w", encoding="utf-8") as f:
    for q in dev:
        scores = denses["section + texte"].scores(q["question"])
        f.write(json.dumps({"question_id": q["question_id"], "partition": "dev",
                            "recherche": f"APP-REC : dense {MODELE_DENSE}, titre de section + texte",
                            "passages_retrouves": [{"passage_id": ids[i], "rang": r, "score": round(s, 4)}
                                                   for r, (i, s) in enumerate(premiers(scores, K), start=1)]},
                           ensure_ascii=False) + "\n")
print(len(dev), "classements denses écrits")

20 classements denses écrits


## Essai de reclassement (2026-10-30)

**Idée.** Au lieu de fusionner des rangs avec un poids à régler, on réunit les candidats des deux recherches puis
un **reclasseur** les trie : `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` (multilingue, 118 M paramètres, CPU). Il lit
la question et le passage ensemble, ce qui est plus précis qu'un cosinus entre deux vecteurs, mais plus lent.

**Configuration fixée avant l'exécution** : candidats = union des 20 premiers BM25 REC-v2 et des 20 premiers denses
(section + texte) ; le reclasseur lit titre de section + texte. Variantes sans paramètre en plus : reclassement des
20 premiers BM25 seuls, ou denses seuls.

**Règle de choix fixée avant l'exécution** : le reclassement n'est retenu que s'il dépasse le Rappel@5 de la
recherche dense seule sans faire perdre de preuve à aucune question.

In [10]:
from src.qa_recherche import Reclasseur, MODELE_RECLASSEUR

debut = time.perf_counter()
reclasseur = Reclasseur()
print(f"{MODELE_RECLASSEUR} chargé en {time.perf_counter() - debut:.1f} s")

def candidats_union(question, n=20):
    cb, cd = classement_bm25(question)[:n], classement_dense(question, "section + texte")[:n]
    return cb + [i for i in cd if i not in cb]

RECLASSEMENT = {
    "BM25 REC-v2": classement_bm25,
    "dense (section + texte)": lambda q: classement_dense(q, "section + texte"),
    "reclassement des 20 premiers BM25": lambda q: reclasseur.reclasser(q, classement_bm25(q)[:20], sections),
    "reclassement des 20 premiers denses": lambda q: reclasseur.reclasser(
        q, classement_dense(q, "section + texte")[:20], sections),
    "reclassement de l'union BM25 + dense": lambda q: reclasseur.reclasser(q, candidats_union(q), sections),
}

lignes, durees = [], {}
for nom, f in RECLASSEMENT.items():
    debut = time.perf_counter()
    for q in dev:
        retrouves = [ids[i] for i in f(q["question"])]
        if q["answerable"]:
            lignes.append({"methode": nom, "question_id": q["question_id"], **mesures_question(q, retrouves, K)})
    durees[nom] = time.perf_counter() - debut
reclasse = pd.DataFrame(lignes)
# Rang réciproque de la 1re preuve (1 si elle est classée première, 0,5 si deuxième…) : mesure l'ordre du top-5.
reclasse["rang_reciproque"] = (1 / reclasse["rang_premiere_preuve"]).fillna(0)
resume_reclasse = reclasse.groupby("methode", sort=False)[
    [f"hit@{K}", f"rappel@{K}", f"couverture_elements@{K}", "rang_reciproque"]].mean().round(3)
resume_reclasse["temps par question (s)"] = (pd.Series(durees) / len(dev)).round(2)
resume_reclasse.to_csv(RES / "recherche_dev_reclassement.csv")
resume_reclasse

cross-encoder/mmarco-mMiniLMv2-L12-H384-v1 chargé en 2.7 s


,hit@5,rappel@5,couverture_elements@5,rang_reciproque,temps par question (s)
methode,,,,,
BM25 REC-v2,0.875,0.781,0.823,0.674,0.00
dense (section + texte),0.938,0.875,0.979,0.702,0.04
reclassement des 20 premiers BM25,0.875,0.750,0.781,0.844,2.06
reclassement des 20 premiers denses,0.938,0.875,0.958,0.853,2.01
reclassement de l'union BM25 + dense,0.938,0.875,0.958,0.853,3.52


In [11]:
# Rang de la première preuve et couverture des éléments par question.
display(reclasse.pivot(index="methode", columns="question_id", values="rang_premiere_preuve").loc[list(RECLASSEMENT)])
reclasse.pivot(index="methode", columns="question_id", values=f"couverture_elements@{K}").loc[list(RECLASSEMENT)].round(2)

question_id,Q001,Q002,Q003,Q004,Q006,Q007,Q008,Q009,Q010,Q011,Q012,Q013,Q014,Q015,Q016,Q017
methode,,,,,,,,,,,,,,,,
BM25 REC-v2,1.0,3.0,1.0,1.0,1.0,1.0,5.0,1.0,2.0,1.0,22.0,1.0,5.0,1.0,140.0,2.0
dense (section + texte),3.0,2.0,2.0,1.0,14.0,1.0,1.0,1.0,3.0,1.0,4.0,1.0,1.0,1.0,4.0,1.0
reclassement des 20 premiers BM25,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,NaN,1.0,1.0,1.0,NaN,1.0
reclassement des 20 premiers denses,1.0,2.0,1.0,1.0,7.0,1.0,1.0,1.0,2.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0
reclassement de l'union BM25 + dense,1.0,2.0,1.0,1.0,7.0,1.0,1.0,1.0,2.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0


question_id,Q001,Q002,Q003,Q004,Q006,Q007,Q008,Q009,Q010,Q011,Q012,Q013,Q014,Q015,Q016,Q017
methode,,,,,,,,,,,,,,,,
BM25 REC-v2,1.00,1.0,1.00,1.0,1.0,1.0,1.0,1.0,0.67,1.0,0.0,1.0,1.0,0.5,0.0,1.0
dense (section + texte),1.00,1.0,0.67,1.0,1.0,1.0,1.0,1.0,1.00,1.0,1.0,1.0,1.0,1.0,1.0,1.0
reclassement des 20 premiers BM25,0.67,1.0,0.67,1.0,1.0,1.0,1.0,1.0,0.67,1.0,0.0,1.0,1.0,0.5,0.0,1.0
reclassement des 20 premiers denses,0.67,1.0,0.67,1.0,1.0,1.0,1.0,1.0,1.00,1.0,1.0,1.0,1.0,1.0,1.0,1.0
reclassement de l'union BM25 + dense,0.67,1.0,0.67,1.0,1.0,1.0,1.0,1.0,1.00,1.0,1.0,1.0,1.0,1.0,1.0,1.0


**Conclusion.** Le reclassement ne fait pas mieux que la recherche dense seule en Rappel@5 (0,875 dans les deux cas) :
**il n'est pas retenu** (règle fixée avant l'essai). Il met bien la première preuve plus souvent en tête (rang
réciproque 0,70 → 0,85), mais fait sortir du top-5 l'une des deux sources divergentes de Q001 et recule Q006, que
BM25 classait premier. Il coûte environ 2 s par question sur CPU contre quelques millisecondes : peu frugal pour
aucun gain mesurable. L'union n'apporte rien par rapport aux 20 premiers denses seuls.